In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
import pickle

In [2]:
with open('./train_data/a2schwa_acc.p', 'rb') as handle:
    data : list[dict] = pickle.load(handle)

In [3]:
data_df = pd.DataFrame(data)
data_df = data_df.fillna('')

In [4]:
X, y = data_df[['desinence', 'new_ending']], data_df['y']

In [5]:
encoder = OneHotEncoder(handle_unknown="ignore")

tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=5
)

model = make_pipeline(encoder, tree)
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('onehotencoder', ...), ('decisiontreeclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['desinence','new_ending']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... ver

In [6]:
model.score(X, y)

0.9151267726686722

In [7]:
import numpy as np
import pandas as pd
from sklearn.tree import _tree

feature_names = encoder.get_feature_names_out(X.columns)

def extract_leaf_rules(tree, feature_names):
    t = tree.tree_
    rules = []

    def walk(node, conditions):
        # Leaf
        if t.feature[node] == _tree.TREE_UNDEFINED:
            predicted_class_index = np.argmax(t.value[node][0])
            predicted_class = tree.classes_[predicted_class_index]

            rules.append({
                "leaf_id": node,
                "conditions": conditions.copy(),
                "predicted_class": predicted_class,
            })
            return

        feature = feature_names[t.feature[node]]
        threshold = t.threshold[node]

        # left branch: <= threshold
        walk(
            t.children_left[node],
            conditions + [(feature, "<=", threshold)]
        )

        # right branch: > threshold
        walk(
            t.children_right[node],
            conditions + [(feature, ">", threshold)]
        )

    walk(0, [])
    return rules

In [8]:
rules = extract_leaf_rules(tree, feature_names)

In [9]:
X_encoded = encoder.transform(X)

leaf_ids = tree.apply(X_encoded) # get leaf for each datum
data_df['leaf_id'] = leaf_ids

In [10]:
reports = []

for rule in rules:
    leaf_id = rule["leaf_id"]
    predicted = rule["predicted_class"]

    subset = data_df[data_df["leaf_id"] == leaf_id].copy()

    subset["correct"] = subset["y"] == predicted

    reports.append({
        "leaf_id": leaf_id,
        "conditions": rule["conditions"],
        "predicted": predicted,
        "n": len(subset),
        "n_correct": subset["correct"].sum(),
        "accuracy": subset["correct"].mean(),
        "correct_words": list(
            subset.loc[
                subset["correct"],
                ["input_index", "output_index"]
            ].itertuples(index=False, name=None)
        ),
        "exceptions": list(
            subset.loc[
                ~subset["correct"],
                ["input_index", "output_index"]
            ].itertuples(index=False, name=None)
        ),
      })

In [11]:
data_dict = {(d['input_index'], d['output_index']):d for d in data_df.to_dict(orient='records')}

In [12]:
def simplify_conditions(conditions, possible_values):
    allowed = {
        feature: set(values)
        for feature, values in possible_values.items()
    }

    # Longest feature names first, in case one feature name
    # happens to be a prefix of another.
    features = sorted(possible_values, key=len, reverse=True)

    for encoded_feature, op, threshold in conditions:

        # Find which original feature this one-hot column belongs to
        feature = next(
            (f for f in features if encoded_feature.startswith(f + "_")),
            None
        )

        if feature is None:
            raise ValueError(
                f"Cannot identify original feature for {encoded_feature!r}"
            )

        value = encoded_feature[len(feature) + 1:]

        # OneHotEncoder produces 0/1 columns, so a split at 0.5 means:
        #
        #   > 0.5   -> feature == value
        #   <= 0.5  -> feature != value

        if op == ">":
            allowed[feature] &= {value}
        elif op == "<=":
            allowed[feature].discard(value)
        else:
            raise ValueError(f"Unexpected operator: {op}")

    # Only show features actually constrained by the rule
    constrained = {
        feature: values
        for feature, values in allowed.items()
        if values != set(possible_values[feature])
    }

    return constrained

In [13]:
reports[0]['conditions']

[('desinence_e', '<=', np.float64(0.5)),
 ('desinence_', '<=', np.float64(0.5)),
 ('new_ending_i', '<=', np.float64(0.5)),
 ('new_ending_uri', '<=', np.float64(0.5))]

In [14]:
possible_values = {
    column: set(X[column].unique())
    for column in X.columns
}

In [15]:
simplify_conditions(reports[0]['conditions'], possible_values)

{'desinence': {'a', 'i', 'ie', 'o', 'u', 'ă'},
 'new_ending': {'e', 'ete', 'ie', 'ii', 'iuri', 'le', 'ouri', 'și'}}

In [19]:
simplify_conditions(reports[4]['conditions'], possible_values)

{'desinence': {''},
 'new_ending': {'e', 'ete', 'ie', 'ii', 'iuri', 'le', 'ouri', 'și'}}